# 🛢️ La Paradoja de la Abundancia
## Petróleo, Instituciones y Desarrollo (1995–2021)

**Proyecto de Portfolio · Análisis de Datos**  
**Dataset:** Global Crude Petroleum Trade 1995–2021 (UN Comtrade / Kaggle)  
**Herramientas:** Python · pandas · plotly · Google Colab

---

### Pregunta central
> *¿Por qué algunos países convierten el petróleo en prosperidad y otros no?*

### Marco teórico
| Concepto | Autor | Año | Nota |
|---|---|---|---|
| Maldición de los Recursos | Richard Auty | 1993 | Concepto fundacional |
| Correlación empírica recursos–crecimiento | Sachs & Warner | 1995 | Confirmación econométrica |
| Instituciones inclusivas vs. extractivas | Acemoglu & Robinson | 2012 | Premio Nobel 2024 |

### ⚠️ Nota metodológica
> **El dataset de UN Comtrade registra exclusivamente comercio declarado oficialmente.**  
> Los volúmenes transportados por la *dark fleet* no aparecen en estas estadísticas.  
> Los 'colapsos' de Venezuela e Irán son colapsos del **comercio formal** — no del volumen real exportado.  
> Este notebook contrasta los datos del dataset con fuentes externas para ofrecer un panorama completo.

---

## 0. Setup — Librerías y configuración

In [32]:
ven_formal = [...]
ven_real = [...]

# Metodología del Análisis

**Dataset principal**
- Fuente: UN Comtrade
- Período: 1995–2021
- Variable: Trade Value (USD)
- Filtro aplicado: Action == "Export"

**Transformaciones**
- Conversión a miles de millones USD (Value_B = Trade Value / 1e9)
- Agrupación por país y año
- Cálculo de participación regional (% sobre total mundial)

**Limitaciones**
- El dataset solo incluye comercio declarado oficialmente.
- Exportaciones bajo sanciones (Irán, Venezuela) pueden estar subestimadas.
- Datos 2022–2024 utilizados en algunas visualizaciones provienen de fuentes externas (EIA, OPEC, UANI, PDVSA).

## Nota sobre estimaciones externas

Los valores "real estimado" para Irán y Venezuela no provienen del dataset UN Comtrade.

Fueron incorporados manualmente a partir de:
- UANI Tanker Tracker
- EIA
- OPEC
- Reuters / Bloomberg

Se utilizan únicamente para ilustrar la divergencia entre comercio formal declarado y exportaciones estimadas reales.

In [33]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

NAVY   = '#1E3A5F'
BLUE   = '#2E6DA4'
GOLD   = '#E8B84B'
RED    = '#C0392B'
GREEN  = '#1A7A4A'
ORANGE = '#D35400'   # color distintivo para dark fleet
TEAL   = '#38BDF8'
TMPL   = 'plotly_dark'

print('✅ Setup completo')

✅ Setup completo


## 1. Carga del dataset



In [34]:
from google.colab import files
uploaded = files.upload()

Saving Global Crude Petroleum Trade 1995-2021.csv to Global Crude Petroleum Trade 1995-2021 (6).csv


In [35]:
df = pd.read_csv('/content/Global Crude Petroleum Trade 1995-2021 (3).csv')

exports = df[df['Action'] == 'Export'].copy()
exports['Value_B'] = exports['Trade Value'] / 1e9

MIDEAST = ['Saudi Arabia','Iraq','Iran','Kuwait','United Arab Emirates','Qatar']
LATAM   = ['Venezuela','Brazil','Colombia','Ecuador','Mexico','Argentina']

print(f'✅ {len(df):,} registros · {df["Year"].min()}–{df["Year"].max()} · {df["Country"].nunique()} países')

✅ 7,925 registros · 1995–2021 · 230 países


## 2. El mapa del poder global
**Arabia Saudita fue #1 los 27 años consecutivos del período.**

In [36]:
top15 = (exports.groupby('Country')['Value_B'].sum()
         .sort_values(ascending=False).head(15).reset_index())
top15['label'] = top15['Value_B'].apply(lambda x: f'${x:,.0f}B')

# Marcar países con dark fleet
dark_fleet_countries = {'Iran', 'Venezuela'}
top15['color'] = top15['Country'].apply(
    lambda c: ORANGE if c in dark_fleet_countries else BLUE
)

fig = go.Figure(go.Bar(
    x=top15.sort_values('Value_B')['Value_B'],
    y=top15.sort_values('Value_B')['Country'],
    orientation='h',
    marker_color=top15.sort_values('Value_B')['color'],
    text=top15.sort_values('Value_B')['label'],
    textposition='outside'
))
fig.add_annotation(x=200, y=1.5, text='⚠ Naranja = dark fleet activa\n(datos subestimados)',
    font=dict(color=ORANGE, size=11), showarrow=False,
    bgcolor='rgba(211,84,0,0.15)', bordercolor=ORANGE, borderwidth=1)
fig.update_layout(
    title='Top 15 Exportadores — Valor Acumulado 1995–2021 (comercio declarado)',
    xaxis_title='Miles de millones USD (comercio formal)',
    template=TMPL, height=500)
fig.show()

print('\n📌 Arabia Saudita: $3.378B acumulados — 46% más que Rusia en segundo lugar')
print('⚠️  Irán (#8) y Venezuela (#10) muestran datos del comercio formal únicamente.')
print('    El comercio real vía dark fleet es sustancialmente mayor (ver sección 4).')


📌 Arabia Saudita: $3.378B acumulados — 46% más que Rusia en segundo lugar
⚠️  Irán (#8) y Venezuela (#10) muestran datos del comercio formal únicamente.
    El comercio real vía dark fleet es sustancialmente mayor (ver sección 4).


In [37]:
# Participación de mercado: Medio Oriente vs. América Latina
world_y = exports.groupby('Year')['Value_B'].sum()
me_y    = exports[exports['Country'].isin(MIDEAST)].groupby('Year')['Value_B'].sum()
la_y    = exports[exports['Country'].isin(LATAM)].groupby('Year')['Value_B'].sum()

share = pd.DataFrame({
    'Medio Oriente %': (me_y/world_y*100).round(1),
    'América Latina %': (la_y/world_y*100).round(1)
}).reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=share['Year'], y=share['Medio Oriente %'],
    name='Medio Oriente', line=dict(color=GOLD, width=2.5),
    fill='tozeroy', fillcolor='rgba(232,184,75,0.1)'))
fig.add_trace(go.Scatter(x=share['Year'], y=share['América Latina %'],
    name='América Latina (comercio formal)', line=dict(color=TEAL, width=2.5),
    fill='tozeroy', fillcolor='rgba(56,189,248,0.1)'))
fig.update_layout(
    title='Participación de Mercado Global (%) — Datos declarados',
    yaxis_title='% del mercado mundial',
    template=TMPL, height=380)
fig.show()

print('\n📌 Medio Oriente: 34% (1995) → 41.5% (2016)')
print('📌 América Latina: 11.4% (1995) → 7.5% (2021)')
print('⚠️  Parte de la caída latinoamericana es ilusión estadística: Venezuela')
print('    migró su comercio al mercado en la sombra, no desapareció del mercado.')


📌 Medio Oriente: 34% (1995) → 41.5% (2016)
📌 América Latina: 11.4% (1995) → 7.5% (2021)
⚠️  Parte de la caída latinoamericana es ilusión estadística: Venezuela
    migró su comercio al mercado en la sombra, no desapareció del mercado.


## 3. La Dark Fleet

### ⚠️ Lo que el dataset no puede medir

El dataset de UN Comtrade registra **comercio declarado**. Venezuela e Irán no dejaron de exportar petróleo — lo que hicieron fue mover su comercio al mercado en la sombra.

**Comparación: datos del dataset vs. estimaciones de fuentes externas**

| País | Dataset 2021 | Estimación real 2021 | Fuente |
|---|---|---|---|
| **Irán** | $130M | ~$35–40B equivalente | UANI / EIA |
| **Venezuela** | $5M | ~$1.1B (crudo + derivados) | OEC / Reuters |
| **Irán (2024)** | N/D | 587M de barriles (+10.75% vs 2023) | UANI |
| **Venezuela (2024)** | N/D | $17.52B (PDVSA reportado) | PDVSA / Reuters |

**El costo del mercado negro:**
- Descuentos de $8–$15 por barril vs. precio de mercado
- Fletes que casi duplican los del mercado regular
- Un único comprador (China) que impone condiciones
- Corrupción: el IRGC captura el 40–50% de ingresos petroleros iraníes (US Treasury / OFAC)
- Activos permanentemente vulnerables a sanciones

**Fuentes:**
- IFMAT (sep. 2024): *Iran's Shadow Oil Trade: The Largest Global Share of Smuggling Vessels*
- UANI Tanker Tracker (metodología AIS + satélite): unitedagainstnucleariran.com
- US Treasury OFAC (dic. 2025): sanciones a redes de transporte de petróleo iraní
- Bloomberg (ene. 2026): *Venezuela's Dark Fleet Emerges After Maduro's Capture*
- Reuters / Reuters Factbox: exportaciones Venezuela 2021–2024
- CNN Business (mar. 2023): *A shadow fleet is helping Russia ship oil*

In [38]:
# Visualización: dataset vs. estimación real — Venezuela e Irán
years = [2018, 2019, 2020, 2021]

# Venezuela
ven_formal  = [36.9, 12.2, 2.6, 0.005]   # dataset
ven_real    = [38.0, 16.0, 4.2, 1.1]     # estimación fuentes externas

# Irán
iran_formal = [55.0, 8.0, 3.5, 0.13]    # dataset
iran_real   = [55.0, 25.0, 15.0, 38.0]  # estimación

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Venezuela: Declarado vs. Real estimado',
                    'Irán: Declarado vs. Real estimado'))

fig.add_trace(go.Bar(x=years, y=ven_formal, name='Dataset formal',
    marker_color=BLUE, opacity=0.7), row=1, col=1)
fig.add_trace(go.Scatter(x=years, y=ven_real, name='Estimación real',
    line=dict(color=ORANGE, width=3, dash='dash'),
    mode='lines+markers', marker=dict(size=10)), row=1, col=1)

fig.add_trace(go.Bar(x=years, y=iran_formal, name='Dataset formal',
    marker_color=BLUE, opacity=0.7, showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=years, y=iran_real, name='Estimación real',
    line=dict(color=ORANGE, width=3, dash='dash'),
    mode='lines+markers', marker=dict(size=10), showlegend=False), row=1, col=2)

fig.update_yaxes(title_text='Miles de millones USD', row=1, col=1)
fig.update_layout(
    title='Dark Fleet: Brecha entre comercio declarado y estimación real (2018–2021)',
    template=TMPL, height=420)
fig.show()

print('\n⚠️  NOTA METODOLÓGICA:')
print('   Las estimaciones "reales" son aproximaciones basadas en:')
print('   - OEC (Observatory of Economic Complexity) para Venezuela')
print('   - UANI Tanker Tracker + EIA para Irán')
print('   No son cifras exactas — son órdenes de magnitud que ilustran la brecha.')


⚠️  NOTA METODOLÓGICA:
   Las estimaciones "reales" son aproximaciones basadas en:
   - OEC (Observatory of Economic Complexity) para Venezuela
   - UANI Tanker Tracker + EIA para Irán
   No son cifras exactas — son órdenes de magnitud que ilustran la brecha.


In [39]:
# Escala de la dark fleet global: contexto
categorias = [
    'Flota global dark fleet (est.)',
    'Sancionados por EEUU/UK/UE',
    'Irán (identificados UANI)',
    'Venezuela (activos 2024)'
]
valores = [1400, 920, 477, 71]
colores = [BLUE, RED, ORANGE, ORANGE]

fig = go.Figure(go.Bar(
    x=categorias, y=valores,
    marker_color=colores,
    text=[f'{v} buques' for v in valores],
    textposition='outside'
))
fig.update_layout(
    title='Escala de la Dark Fleet Global — Contexto (estimaciones 2024)',
    yaxis_title='Número de buques',
    template=TMPL, height=400)
fig.show()

print('\n📌 La dark fleet global supera los 1.400 buques.')
print('📌 Más de 920 están sancionados por al menos un gobierno occidental.')
print('📌 La infraestructura se originó para Venezuela e Irán y creció con las sanciones a Rusia (2022+).')
print('\nFuentes: UANI (2024), Reuters, Bloomberg, CNN Business')


📌 La dark fleet global supera los 1.400 buques.
📌 Más de 920 están sancionados por al menos un gobierno occidental.
📌 La infraestructura se originó para Venezuela e Irán y creció con las sanciones a Rusia (2022+).

Fuentes: UANI (2024), Reuters, Bloomberg, CNN Business


## 4. Casos de colapso: Venezuela e Irán

Con el contexto de la dark fleet incorporado, los casos adquieren mayor profundidad:
no son solo colapsos — son **colapsos institucionales que desplazaron el comercio formal al mercado negro**.

In [40]:
# Venezuela: colapso formal + estimación real
ven = exports[exports['Country']=='Venezuela'].sort_values('Year')

# Datos externos post-dataset
ven_ext_years = [2022, 2023, 2024]
ven_ext_vals  = [4.0, 4.05, 17.52]   # OEC + PDVSA

fig = go.Figure()
fig.add_trace(go.Bar(x=ven['Year'], y=ven['Value_B'],
    name='Declarado (dataset)', marker_color=BLUE, opacity=0.8))
fig.add_trace(go.Scatter(x=ven_ext_years, y=ven_ext_vals,
    name='Real estimado post-dataset (PDVSA/OEC)',
    line=dict(color=ORANGE, width=3, dash='dash'),
    mode='lines+markers', marker=dict(size=10)))
fig.add_annotation(x=2013, y=84, text='Pico declarado: $81.6B',
    font=dict(color=GOLD, size=11), showarrow=False)
fig.add_annotation(x=2019, y=18, text='Sanciones EEUU (2019)',
    font=dict(color=RED, size=11), showarrow=True, arrowhead=2, arrowcolor=RED)
fig.add_annotation(x=2021, y=8, text='Dataset: $5M\nReal: ~$1.1B',
    font=dict(color=ORANGE, size=11), showarrow=True, ax=40, ay=-40)
fig.add_vrect(x0=2021.5, x1=2024.5, fillcolor='rgba(211,84,0,0.06)',
    line_width=0, annotation_text='Post-dataset\n(fuentes externas)', annotation_position='top left')
fig.update_layout(
    title='Venezuela — Colapso del mercado formal + estimación real 1995–2024',
    yaxis_title='Miles de millones USD', template=TMPL, height=430)
fig.show()

print('\n📌 CASO: Instituciones extractivas (Acemoglu & Robinson)')
print('   PDVSA fue vaciada de técnicos y usada para fines políticos.')
print('   Reservas #1 del mundo → $5M declarados en 2021 → ~$1.1B real (dark fleet).')
print('   2024: $17.52B reportado por PDVSA — 71 supertanqueros hacia China (Bloomberg, ene. 2026).')


📌 CASO: Instituciones extractivas (Acemoglu & Robinson)
   PDVSA fue vaciada de técnicos y usada para fines políticos.
   Reservas #1 del mundo → $5M declarados en 2021 → ~$1.1B real (dark fleet).
   2024: $17.52B reportado por PDVSA — 71 supertanqueros hacia China (Bloomberg, ene. 2026).


In [41]:
# Irán: colapso formal + volumen real estimado
iran = exports[exports['Country']=='Iran'].sort_values('Year')

iran_real_years = [2022, 2023, 2024]
iran_real_vals  = [35.0, 40.0, 44.0]  # estimación UANI (587M bbl * ~$75)

fig = go.Figure()
fig.add_trace(go.Bar(x=iran['Year'], y=iran['Value_B'],
    name='Declarado (dataset)', marker_color='#9B59B6', opacity=0.8))
fig.add_trace(go.Scatter(x=iran_real_years, y=iran_real_vals,
    name='Real estimado post-dataset (UANI)',
    line=dict(color=ORANGE, width=3, dash='dash'),
    mode='lines+markers', marker=dict(size=10)))
fig.add_annotation(x=2011, y=92, text='Pico declarado: $87.2B',
    font=dict(color=GOLD, size=11), showarrow=False)
fig.add_annotation(x=2012, y=60, text='Sanciones JCPOA',
    font=dict(color=RED, size=11), showarrow=True, ax=-60, ay=-30)
fig.add_annotation(x=2018, y=55, text='Reimposición Trump',
    font=dict(color=RED, size=11), showarrow=True, ax=50, ay=-30)
fig.add_annotation(x=2021, y=15, text='Dataset: $130M\nReal: ~$35–40B',
    font=dict(color=ORANGE, size=11), showarrow=True, ax=40, ay=-30)
fig.add_vrect(x0=2021.5, x1=2024.5, fillcolor='rgba(211,84,0,0.06)',
    line_width=0, annotation_text='Post-dataset\n(UANI)', annotation_position='top left')
fig.update_layout(
    title='Irán — Colapso del mercado formal + estimación real vía dark fleet',
    yaxis_title='Miles de millones USD', template=TMPL, height=430)
fig.show()

print('\n📌 CASO: Aislamiento geopolítico + captura institucional del IRGC')
print('   Dataset 2021: $130M. Estimación real: ~$35–40B (UANI/EIA).')
print('   2023: 533M de barriles exportados. 2024: 587M (UANI).')
print('   IRGC captura 40–50% de ingresos petroleros iraníes (US Treasury / OFAC).')
print('   Fuente IFMAT (sep. 2024): 1.7M barriles/día vía ghost fleet.')


📌 CASO: Aislamiento geopolítico + captura institucional del IRGC
   Dataset 2021: $130M. Estimación real: ~$35–40B (UANI/EIA).
   2023: 533M de barriles exportados. 2024: 587M (UANI).
   IRGC captura 40–50% de ingresos petroleros iraníes (US Treasury / OFAC).
   Fuente IFMAT (sep. 2024): 1.7M barriles/día vía ghost fleet.


## Impacto de sanciones sobre exportaciones formales
Análisis de correlación simple entre período sancionado y caída en exportaciones declaradas.

In [42]:
# Crear variable dummy de sanciones
exports['Sanction_Iran'] = np.where(
    (exports['Country'] == 'Iran') & (exports['Year'] >= 2018),
    1, 0
)

exports['Sanction_Venezuela'] = np.where(
    (exports['Country'] == 'Venezuela') & (exports['Year'] >= 2019),
    1, 0
)

# Filtrar solo Irán y Venezuela
sanctions_df = exports[exports['Country'].isin(['Iran','Venezuela'])].copy()

# Correlación entre sanción y exportaciones formales
corr_iran = sanctions_df[sanctions_df['Country']=='Iran'][['Sanction_Iran','Value_B']].corr().iloc[0,1]
corr_ven  = sanctions_df[sanctions_df['Country']=='Venezuela'][['Sanction_Venezuela','Value_B']].corr().iloc[0,1]

print("Correlación sanciones - exportaciones formales")
print(f"Irán: {corr_iran:.2f}")
print(f"Venezuela: {corr_ven:.2f}")

Correlación sanciones - exportaciones formales
Irán: -0.36
Venezuela: -0.38


La correlación negativa entre período sancionado y exportaciones formales (-0.36 en Irán y -0.38 en Venezuela) confirma una relación inversa consistente. Sin embargo, la magnitud moderada sugiere que las sanciones actúan como acelerador de un deterioro institucional y productivo previo, más que como causa única del colapso formal

## 5. El camino alternativo: Brasil, Argentina e Irak
**El mismo tipo de recurso, gestionado con instituciones funcionales, produce resultados radicalmente distintos.**

In [43]:
# Brasil: el crecimiento más extraordinario del dataset
bra = exports[exports['Country']=='Brazil'].sort_values('Year')

fig = go.Figure(go.Bar(
    x=bra['Year'], y=bra['Value_B'],
    marker_color=[GREEN if y >= 2006 else TEAL for y in bra['Year']]
))
fig.add_vline(x=2005.5, line_dash='dash', line_color='white', opacity=0.3)
fig.add_annotation(x=2003, y=4, text='Descubrimiento pre-sal (2006)',
    font=dict(color=GOLD, size=11), showarrow=False)
fig.add_annotation(x=2021, y=32, text='Pico: $30.7B (2021)',
    font=dict(color=GREEN, size=11), showarrow=False)
fig.update_layout(
    title='Brasil — $60M (1995) → $30.7B (2021): Apertura + inversión tecnológica',
    yaxis_title='Miles de millones USD', template=TMPL, height=400)
fig.show()

print('\n📌 CASO: Apertura + inversión tecnológica (pre-sal offshore)')
print('   $60M (1995) → $30.7B (2021): crecimiento 500x en 26 años.')
print('   Único país del grupo en pico histórico al cierre del dataset.')
print('   Petrobras + capitales privados internacionales = instituciones inclusivas en acción.')


📌 CASO: Apertura + inversión tecnológica (pre-sal offshore)
   $60M (1995) → $30.7B (2021): crecimiento 500x en 26 años.
   Único país del grupo en pico histórico al cierre del dataset.
   Petrobras + capitales privados internacionales = instituciones inclusivas en acción.


In [44]:
# Argentina + Venezuela: la contracara directa
arg = exports[exports['Country']=='Argentina'].sort_values('Year')
ven2 = exports[exports['Country']=='Venezuela'].sort_values('Year')

# Post-dataset Argentina (Vaca Muerta)
arg_vm_years = [2022, 2023, 2024]
arg_vm_vals  = [2.1, 3.8, 5.5]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Argentina — Vaca Muerta post-2021', 'Venezuela — Dark fleet post-2021'))

fig.add_trace(go.Bar(x=arg['Year'], y=arg['Value_B'],
    name='Argentina (dataset)', marker_color=TEAL, opacity=0.8), row=1, col=1)
fig.add_trace(go.Scatter(x=arg_vm_years, y=arg_vm_vals,
    name='Argentina Vaca Muerta (est.)',
    line=dict(color=GREEN, width=3, dash='dash'),
    mode='lines+markers', marker=dict(size=9)), row=1, col=1)

fig.add_trace(go.Bar(x=ven2['Year'], y=ven2['Value_B'],
    name='Venezuela (dataset)', marker_color=RED, opacity=0.7, showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=[2022,2023,2024], y=[4.0,4.05,17.52],
    name='Venezuela real (PDVSA/OEC)',
    line=dict(color=ORANGE, width=3, dash='dash'),
    mode='lines+markers', marker=dict(size=9), showlegend=False), row=1, col=2)

fig.update_layout(title='Argentina vs. Venezuela — Dos trayectorias post-2021',
    template=TMPL, height=420)
fig.show()

print('\n📌 ARGENTINA post-2021 (Vaca Muerta):')
print('   +33%/año entre 2017–2023 | 187k bbl/día en 2024 (récord 20 años)')
print('   Superávit energético 2024: $5.688B | Proyección 2030: triplicar exportaciones')
print('   Fuente: Secretaría de Energía Argentina / IAPG (2024)')
print('\n⚠️  VENEZUELA post-2021 (dark fleet):')
print('   $17.52B reportado por PDVSA en 2024 — pero vía 71 supertanqueros hacia China')
print('   Sin acceso al mercado formal, a precio castigado, con un único comprador')


📌 ARGENTINA post-2021 (Vaca Muerta):
   +33%/año entre 2017–2023 | 187k bbl/día en 2024 (récord 20 años)
   Superávit energético 2024: $5.688B | Proyección 2030: triplicar exportaciones
   Fuente: Secretaría de Energía Argentina / IAPG (2024)

⚠️  VENEZUELA post-2021 (dark fleet):
   $17.52B reportado por PDVSA en 2024 — pero vía 71 supertanqueros hacia China
   Sin acceso al mercado formal, a precio castigado, con un único comprador


## 6. Análisis de tendencias (mercado formal)
Regresión lineal 2015–2021 sobre datos declarados.

In [45]:
resultados = []
for pais in LATAM + MIDEAST:
    d = exports[(exports['Country']==pais) & (exports['Year']>=2015)].sort_values('Year')
    if len(d) >= 4:
        coef = np.polyfit(d['Year'], d['Value_B'], 1)
        v19 = exports[(exports['Country']==pais) & (exports['Year']==2019)]['Value_B'].values
        v21 = exports[(exports['Country']==pais) & (exports['Year']==2021)]['Value_B'].values
        mom = ((v21[0]-v19[0])/v19[0]*100) if len(v19) and len(v21) and v19[0]>0 else None
        dark_flag = '⚠️ dark fleet' if pais in ['Venezuela','Iran'] else '✅ mercado formal'
        resultados.append({
            'País': pais,
            'Región': 'América Latina' if pais in LATAM else 'Medio Oriente',
            'Tendencia B/año': round(coef[0], 2),
            'Momentum 19→21 %': round(mom, 1) if mom else 'N/D',
            'Estado': dark_flag
        })

tend_df = pd.DataFrame(resultados).sort_values('Tendencia B/año', ascending=False)

colors = tend_df['Tendencia B/año'].apply(
    lambda x: RED if x < -2 else (ORANGE if x < 0 else GREEN)
).tolist()

fig = go.Figure(go.Bar(
    x=tend_df['Tendencia B/año'],
    y=tend_df['País'] + ' ' + tend_df['Estado'],
    orientation='h',
    marker_color=colors
))
fig.add_vline(x=0, line_color='white', opacity=0.4)
fig.update_layout(
    title='Tendencia Exportaciones 2015–2021 — Datos declarados (B USD/año)',
    template=TMPL, height=500)
fig.show()

print('\n⚠️  Venezuela e Irán muestran tendencia negativa en el mercado formal.')
print('    Esto refleja el colapso del comercio declarado, no necesariamente del volumen real.')
print(tend_df.to_string(index=False))


⚠️  Venezuela e Irán muestran tendencia negativa en el mercado formal.
    Esto refleja el colapso del comercio declarado, no necesariamente del volumen real.
                País         Región  Tendencia B/año  Momentum 19→21 %           Estado
                Iraq  Medio Oriente             3.38              -3.6 ✅ mercado formal
              Brazil América Latina             2.88              26.5 ✅ mercado formal
        Saudi Arabia  Medio Oriente             2.77              -7.8 ✅ mercado formal
United Arab Emirates  Medio Oriente             2.05               1.3 ✅ mercado formal
              Kuwait  Medio Oriente             1.03              -4.2 ✅ mercado formal
              Mexico América Latina             0.52             -23.8 ✅ mercado formal
             Ecuador América Latina             0.15              -4.9 ✅ mercado formal
           Argentina América Latina             0.07              -9.0 ✅ mercado formal
            Colombia América Latina            -

## 7. Conclusiones

### Lo que los datos confirman

| Lección | Evidencia |
|---|---|
| **El petróleo sin instituciones no desaparece — migra al mercado negro** | Venezuela e Irán: colapso formal → dark fleet activa |
| **La dark fleet es el costo real del colapso institucional** | Descuento $8–15/bbl, fletes dobles, un solo comprador |
| **La apertura e inversión generan el camino alternativo** | Brasil: $60M → $30.7B | Irak: $28M → $72B |
| **El potencial de LatAm está en el no convencional** | Argentina: Vaca Muerta +33%/año post-2021 |

### Referencias principales
- **Auty (1993):** *Sustaining Development in Mineral Economies* — Maldición de los Recursos
- **Sachs & Warner (1995):** *Natural Resource Abundance and Economic Growth* — NBER
- **Acemoglu & Robinson (2012):** *Why Nations Fail* — Premio Nobel Economía 2024
- **IFMAT (sep. 2024):** *Iran's Shadow Oil Trade: The Largest Global Share of Smuggling Vessels*
- **UANI Tanker Tracker:** seguimiento AIS + satélite de exportaciones iraníes
- **US Treasury OFAC (dic. 2025):** sanciones a redes dark fleet iraní
- **Bloomberg (ene. 2026):** *Venezuela's Dark Fleet Emerges After Maduro's Capture*
- **Reuters Factbox (2025):** exportaciones Venezuela 2021–2024
- **Secretaría de Energía Argentina / IAPG (2024):** estadísticas Vaca Muerta

---
*Proyecto de Portfolio · Análisis de Datos · 2025 · Python + Google Colab*